<a href="https://colab.research.google.com/github/ManoloVM21/MachineLearningTeam4/blob/main/Cleaned_RF_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Team 4: Random Forest Model


In [51]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline


# Reload clean copy of the data (or use your existing 'campaign' if it is untouched)
bank_data = pd.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bank.csv')

bank_data = bank_data.dropna().copy()

bank_data['y'].value_counts(normalize=True)

bank_data_filtered = bank_data.drop(columns = ["cons.conf.idx", "emp.var.rate", "education", "marital"])


In [31]:
bank_data

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37064,73,retired,married,professional.course,no,yes,no,cellular,nov,fri,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes
37065,46,blue-collar,married,professional.course,no,no,no,cellular,nov,fri,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
37066,56,retired,married,university.degree,no,yes,no,cellular,nov,fri,2,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
37067,44,technician,married,professional.course,no,no,no,cellular,nov,fri,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes


Define features X and target y

In [32]:
X = bank_data.drop(columns = 'y')
y = bank_data['y']

In [33]:
categorical_cols = X.select_dtypes(include=['object']).columns
numeric_cols = X.select_dtypes(exclude=['object']).columns

Preprocessing Process

In [34]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)


Over Sampler

In [52]:
ros = RandomOverSampler(
    sampling_strategy='auto',  # balance minority to majority
    random_state=42
)


Random Forest

In [61]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    class_weight= 'balanced',# 🔑 VERY important here
    n_jobs=-1
)


Pipelines

In [62]:
#Without Over Sampling
model = Pipeline(steps=[
   ('preprocess', preprocessor),
   ('rf', rf)
])


In [ ]:


#With Over Sampling
model = ImbPipeline(steps=[
    ('preprocess', preprocessor),
    ('oversample', ros),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])


Train Test Split

In [63]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)


Fit the Model

In [64]:
model.fit(X_train, y_train)


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
       'month', 'day_of_week', 'poutcome'],
      dtype='object')),
                                                 ('num', 'passthrough',
                                                  Index(['age', 'campaign', 'pdays', 'previous', 'emp.var.rate',
       'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed'],
      dtype='object'))])),
                ('rf',
                 RandomForestClassifier(class_weight='balanced',
                                        min_samples_leaf=5,
                                        min_samples_split=10, n_estimators=300,
                                        n_jobs=-1, random_state=42))])

Evaluation

In [65]:
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

          no       0.95      0.90      0.92      8216
         yes       0.44      0.61      0.51      1052

    accuracy                           0.87      9268
   macro avg       0.70      0.75      0.72      9268
weighted avg       0.89      0.87      0.88      9268

[[7417  799]
 [ 414  638]]


Feature Importance

In [58]:
feature_names = (
    model.named_steps['preprocess']
    .get_feature_names_out()
)

importances = model.named_steps['rf'].feature_importances_

feat_imp = (
    pd.Series(importances, index=feature_names)
    .sort_values(ascending=False)
    .head(15)
)

feat_imp


,0
num__age,0.139307
num__euribor3m,0.124212
num__campaign,0.078162
num__nr.employed,0.063274
num__emp.var.rate,0.050632
num__cons.conf.idx,0.027223
num__cons.price.idx,0.025006
num__pdays,0.020275
cat__housing_no,0.018660
cat__housing_yes,0.018391


Hold Out Data

In [49]:
holdout = pd.read_csv("https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bank_holdout_test_mini.csv")

holdout_preds = model.predict(holdout)
holdout_probs = model.predict_proba(holdout)[:, 1]  # probability of class 1 ("yes")


In [50]:
# Just the predicted class (0/1)
preds_only = pd.DataFrame({"predictions": holdout_preds})
preds_only.to_csv("team4-module2-predictions.csv", index=False)
